# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
%pip install -q --upgrade pypdf gradio openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [11]:
from pathlib import Path
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Extrae y concatena el texto de todas las páginas de un PDF."""
    reader = PdfReader(str(pdf_path))
    return "\n\n".join(page.extract_text() or "" for page in reader.pages)

search_roots = [Path.cwd(), *Path.cwd().parents]
pdf_candidates = [
    root / filename
    for root in search_roots
    for filename in (
        "attention_is_all_you_need.pdf",
        "1706.03762v7.pdf",
        "notebooks/Lecture9/1706.03762v7.pdf",
    )
]
pdf_path = next((path for path in pdf_candidates if path.is_file()), None)
if pdf_path is None:
    raise FileNotFoundError(
        "junto al notebook o usa notebooks/Lecture9/1706.03762v7.pdf."
    )

document_text = extract_text_from_pdf(pdf_path)
print(f"PDF: {pdf_path} | Caracteres extraídos: {len(document_text):,}")
print(document_text[:500])

PDF: c:\General\Eafit\Semestre8\IA\si3003-artificial-intelligence\notebooks\Lecture9\1706.03762v7.pdf | Caracteres extraídos: 39,613
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [12]:
import os
from pathlib import Path
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI, APIError, AuthenticationError

# Busca el .env desde la carpeta actual hasta las carpetas superiores.
env_file = next(
    (folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / ".env").is_file()),
    None,
)
if env_file is not None:
    load_dotenv(env_file, override=True)

api_key = os.getenv("NVIDIA_API_KEY", "").strip().strip('"').strip("'")
if not api_key:
    api_key = getpass("Introduce tu NVIDIA_API_KEY: ").strip()
if not api_key:
    raise RuntimeError(
        "No se encontr? NVIDIA_API_KEY. Config?rala en un archivo .env."
    )

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
)
MODELO = "nvidia/nemotron-3-super-120b-a12b"


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [13]:
def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper 'Attention Is All You Need'.
Responde ?nicamente usando la informaci?n del documento incluido abajo.
Si la respuesta no aparece en el documento, responde exactamente:
"No encuentro esa informaci?n en el documento".
No uses conocimiento externo ni inventes datos.

--- INICIO DEL DOCUMENTO ---
{document_text}
--- FIN DEL DOCUMENTO ---"""

def get_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in content
        )
    return str(content)

def history_to_messages(history):
    messages = []
    for entry in history or []:
        if isinstance(entry, dict):
            role = entry.get("role")
            if role in {"user", "assistant"}:
                messages.append({"role": role, "content": get_text(entry.get("content", ""))})
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_message, assistant_message = entry
            messages.append({"role": "user", "content": get_text(user_message)})
            if assistant_message:
                messages.append({"role": "assistant", "content": get_text(assistant_message)})
    return messages

def chat_con_documento(message, history, document_text):
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    try:
        response = client.chat.completions.create(
            model=MODELO,
            messages=messages,
            temperature=1.0,
            top_p=0.95,
            stream=True,
        )
        answer = ""
        for chunk in response:
            if not chunk.choices:
                continue
            content = chunk.choices[0].delta.content
            if content:
                answer += content
                yield answer
    except AuthenticationError:
        message = (
            "Error 401: NVIDIA rechaz? la API key. Verifica que sea una clave "
            "activa de NVIDIA Build, que est? en el archivo .env y reinicia el kernel."
        )
        print(message)
        yield message
    except APIError as error:
        message = f"Error de NVIDIA NIM ({error.status_code or 'API'}): {error}"
        print(message)
        yield message


## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [15]:
import gradio as gr

reader = PdfReader(str(pdf_path))
print(f"Páginas: {len(reader.pages)} | Caracteres: {len(document_text):,}")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    title="Asistente sobre Attention Is All You Need",
    description="Responde preguntas usando únicamente el texto extraído del paper.",
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Qué es el mecanismo de atención?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
    ],
)

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    show_error=True,
)

Páginas: 15 | Caracteres: 39,613
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

## Respuestas

1. **Limitación de enviar siempre el documento completo.** Cada pregunta vuelve a enviar y procesar todo el texto del PDF, aunque solo sea necesario un fragmento. Esto aumenta el consumo de tokens, el costo y el tiempo de respuesta. Además, un documento grande puede superar la ventana de contexto del modelo.

2. **Por qué todavía no es RAG.** En este ejercicio no existe una etapa de recuperación. La aplicación extrae todo el PDF y lo incluye directamente en el `system prompt` antes de cada consulta. Un sistema RAG dividiría el documento en fragmentos, buscaría los más relevantes para la pregunta y enviaría únicamente esos fragmentos al modelo.

3. **Cómo comprobar el origen de la respuesta.** Se deben hacer preguntas cuya respuesta aparezca en el paper y preguntas trampa sobre temas ausentes, como `¿Qué es GPT-4?`. El modelo debería responder `No encuentro esa información en el documento`. También se puede comparar la respuesta con el PDF, exigir citas o números de página y repetir la prueba retirando el documento del prompt para observar si la respuesta cambia. Estas pruebas aumentan la confianza, aunque el prompt por sí solo no garantiza completamente que el modelo no use conocimiento previo.

4. **Riesgos de insertar documentos de usuarios directamente en el prompt.** El documento puede contener instrucciones maliciosas (*prompt injection*), datos personales o información confidencial. También puede ser demasiado grande, aumentar la latencia y el costo, contener errores o información desactualizada, y producir respuestas incorrectas. Como el texto se envía a una API externa, aparecen además riesgos de privacidad y de exposición de datos. Finalmente, al no recuperar fragmentos específicos, es más difícil demostrar qué parte del documento sustenta la respuesta.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)